# In-Bed Pose Estimation: Remote GPU Runner (Kaggle Dual Tesla T4)

This notebook automates distributed dual-GPU training (`DDP`), Optuna Hyperparameter Optimization (HPO), and Tryhard analysis for **inbed-pose-mlops** on Kaggle.

> **Accelerator Setup**: In Kaggle notebook settings (right sidebar), select **Accelerator: GPU T4 x2**.

In [ ]:
# 1. Environment Verification
!nvidia-smi
import torch

print("CUDA Available:", torch.cuda.is_available())
print("Device Count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# 2. Clone Repository & Install Dependencies
!git clone https://github.com/MichalRedm/inbed-pose-mlops.git
%cd inbed-pose-mlops
!pip install -r requirements.txt

In [ ]:
# 3. Locate SLP Dataset
from pathlib import Path

possible_paths = [
    Path("/kaggle/input/slp-dataset/danaLab"),
    Path("/kaggle/input/simultaneously-collected-multimodal-lying-pose/danaLab"),
    Path("data/SLP/danaLab"),
]
data_dir = next((p for p in possible_paths if p.exists()), None)
print(f"Resolved SLP dataset directory: {data_dir}")

In [ ]:
# 4. Train Dual-GPU DDP Baseline Model
!python scripts/train.py \
    --config configs/remote_kaggle.yaml \
    --data_dir {data_dir} \
    --devices 2 \
    --strategy ddp \
    --batch_size 32 \
    --max_epochs 30

In [ ]:
# 5. Run Optuna Hyperparameter Optimization Study
!python scripts/tune.py \
    --config configs/hpo.yaml \
    --data_dir {data_dir} \
    --n_trials 20 \
    --max_epochs 15 \
    --baseline_score 0.72

In [ ]:
# 6. Analyze HPO Study & Generate Tryhard Visualizations
!python scripts/analyze_hpo.py \
    --config configs/hpo.yaml \
    --baseline_score 0.72 \
    --output_dir artifacts/hpo

In [ ]:
# 7. Render Generated Artifacts Inline
from IPython.display import Image, display

for plot_name in ["optimization_history.png", "param_importances.png"]:
    plot_path = Path("artifacts/hpo") / plot_name
    if plot_path.exists():
        print(f"=== {plot_name} ===")
        display(Image(filename=str(plot_path)))

In [ ]:
# 8. Package Results for Download
!tar -czvf /kaggle/working/inbed_pose_results.tar.gz artifacts/ kaggle_mlflow.db optuna_study.db
print("Exported archive to /kaggle/working/inbed_pose_results.tar.gz")